## Notebook 2

This notebook explores two strategies to improve upon the baseline results from Notebook 1. 

**Part 1 : Better fine-tuning strategy.** The default fine-tuning setup is not well suited for a small dataset like 
SHU-MI, which leads to quick overfitting as shown in Notebook 1. We explore different fine-tuning strategy : freezing the backbone, adding dropout, dataset augmentation and using LoRA adapters on the attention mechanism.

**Part 2 : Architectural improvement.** Propose a new spatial encoding based on the 3D coordinates of the electrodes on the scalp.

## Part 1

There is a clear discrepancy between the tensor shapes seen during pre-training 
and fine-tuning:

| Split        | Shape                  |
|--------------|------------------------|
| Pre-training | [64, 19, 30, 200]      |
| Fine-tuning  | [64, 32,  4, 200]      |

Both the number of channels (19 vs 32) and the number of temporal patches 
(30 vs 4) are different. 

ACPE module uses a depthwise convolution with a fixed kernel of size (19, 7), trained to encode positions over a (19 channels × 30 patches) grid. When applied to a (32 × 4) grid, the kernel operates almost entirely on padding.
Also, the dataset is split across individuals: subjects 1–15 for training, 16–20 for validation, and 21–25 for test. The EEG signals vary between individuals in terms of amplitude, frequency profile, and spatial distribution of activity. This inter-subject variability introduces a domain gap between splits.

Also, we have very little training data, compared to the number of parameters in the model.

| Classifier           | Trainable params | % of total |
|----------------------|-----------------|------------|
| all_patch_reps       | 25,524,801      | 96.8%      |
| all_patch_reps_twolayer | 10,004,001   | 39.6%      |
| all_patch_reps_onelayer | 4,883,600    | 19.3%      |
| avgpooling_patch_reps |         201    |  0.001%    |

The default classifier contains more trainable parameters than the entire backbone. For a dataset of only 7K training samples, this is a fundamental mismatch : the model has more than twice as many parameters as training examples, making overfitting almost inevitable regardless of any other design choice.

The training dataset is balanced, with 3602 for class 0 and 3608 for class 1.

### Fine-tuning strategy investigation


All strategies suffer from the same fundamental issue : too many parameters are updated. LoRA addresses this directly by constraining updates to a low-rank subspace, only ~115K parameters are trainable in the attention layers. I also changed the classifier to all_patchs_reps_onelayer when training LoRA to have fewer parameters to train.

And this does reduce the val/test gap while having the best absolute performance.This confirms that the main challenge on SHU-MI is not model capacity but generalization across subjects. LoRa allows us to have better representations.

Unfreezing the positional encoding, alongside LoRA allowed us to increase the performances even more. It resulted in a very healthier fine-tuning as we can see on the metrics graph (in the figure folder).


| Strategy | Test Acc | ROC-AUC | Val/Test gap |
|----------|----------|---------|--------------|
| Baseline | 0.6150 | 0.6813 | ~4 pts |
| Frozen backbone | 0.6002 | 0.6617 | ~1 pt |
| Gradual unfreezing | 0.6231 | 0.6926 | ~3 pts |
| Early stopping | 0.6240 | 0.6962 | ~2.5 pts |
| LoRA only | 0.6094 | 0.6638 | ~0.4 pts |
| **LoRA + PE** | **0.6311** | **0.6910** | **~0.5 pts** |
| LoRA + PE + patch_emb | 0.6110 | 0.6629 | ~2.5 pts |
| LoRA + PE + Data Augmentation | 0.6276 | 0.6915 | ~1 pts |


This is the code to reproduce the results with LoRA adaptaters :

In [ ]:
from utils.llora import apply_lora_attention
from datasets import shu_dataset
from models import model_for_shu
from finetune_trainer import Trainer
from utils.util import Params
import torch
import torch.nn as nn

params = Params()  ## default fine-tuning parameters of the paper

load_dataset = shu_dataset.LoadDataset(params)
data_loader = load_dataset.get_data_loader()

params.classifier = 'all_patch_reps_onelayer'
model_lora = model_for_shu.Model(params)
model_lora = apply_lora_attention(model_lora, positional_encoding = True)

t = Trainer(params, data_loader, model_lora)
t.train_for_binaryclass()

This is the code for Data Augmentation : 

In [ ]:
from utils.util import augment_dataset

X_train, y_train = [], []
for batch in data_loader['train']:
    data, labels = batch
    X_train.append(data)
    y_train.append(labels)

X_train = torch.cat(X_train, dim=0) 
y_train = torch.cat(y_train, dim=0)

print(f"Avant augmentation : {X_train.shape[0]} samples")

X_aug, y_aug = augment_dataset(X_train, y_train, n_augments=2)

print(f"Après augmentation : {X_aug.shape[0]} samples")

aug_dataset    = torch.utils.data.TensorDataset(X_aug, y_aug)
aug_dataloader = torch.utils.data.DataLoader(aug_dataset, batch_size=params.batch_size, shuffle=True)

data_loader['train'] = aug_dataloader

What is important to notice is also that Data Augmentation doesn't improve the performances. So the performance gains are limited by the structural incompatibility between the pretraining and fine-tuning configurations, not by the fine-tuning strategy itself. 

Closing the remaining performance gap would require addressing architectural changes at pretraining time. 

## Part 2 : Positional Embedding

Going into the SHU-MI dataset paper, we have access to the electrodes used and to the montage. Since the dataset is already processed, I'll make the hypothesis that the electrodes are in order.

For testing the effects of this new spatial encoding, we would have to do a careful ablation study on the pretraining.
I decided to keep the temporal encoding as a 1D convolution on the electrode's grid (noted CPE in the paper), but it has to be tested. Maybe an absolute positional encoding (sinusoidal) combined with this spatial encoding improve performance while having less parameters. 

The temporal embedding remains a problem for datasets with fewer patchs, such as SHU-MI or SEED-V. We would need to do an extensive review of the litterature, many papers introduce relative temporal embedding or rotative positional encoding.

Of course, it involves putting priors into the model, and involves that for each dataset, have to carefully look at the electrodes position and carefully look at the input. But I think the added value could be great and it is much more explainable and cognitively meaningful.

In [4]:
import mne

montage = mne.channels.make_standard_montage('standard_1020')
positions = montage.get_positions()
coords_3d = positions['ch_pos'] 

In [ ]:
standard_32 = [
    'Fp1', 'Fp2',
    'F7', 'F3', 'Fz', 'F4', 'F8',
    'FC5', 'FC1', 'FC2', 'FC6',
    'A1', 'T3', 'C3', 'Cz', 'C4', 'T4','A2',
    'CP5', 'CP1', 'CP2', 'CP6',
    'T5', 'P3', 'Pz', 'P4', 'T6',
    'PO3', 'PO4', 
    'O1', 'Oz', 'O2'
]

coords = torch.tensor(
    [positions['ch_pos'][ch] for ch in standard_32],
    dtype=torch.float32
)

In [10]:
class SpatialEncoding(nn.Module):
    def __init__(self, d_model, coords):
        super().__init__()
        self.register_buffer('coords', coords)
        
        self.mlp = nn.Sequential(
            nn.Linear(3, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
    
    def forward(self, ch_num):
        return self.mlp(self.coords[:ch_num]).unsqueeze(0).unsqueeze(2)   ##(1, ch_num, 1, d_model)

In [ ]:
class CoordsPatchEmbedding(nn.Module):
    def __init__(self, in_dim, out_dim, d_model, seq_len, coords):
        super().__init__()
        self.d_model = d_model
        
        self.temporal_encoding = nn.Sequential(
            nn.Conv2d(in_channels=d_model, out_channels=d_model,
                      kernel_size=(1, 7), stride=(1, 1), padding=(0, 3),
                      groups=d_model),
        )
        
        self.spatial_encoding = SpatialEncoding(d_model, coords)
        
        self.mask_encoding = nn.Parameter(torch.zeros(in_dim), requires_grad=False)
    
        self.proj_in = nn.Sequential(
                nn.Conv2d(in_channels=1, out_channels=25, kernel_size=(1, 49), stride=(1, 25), padding=(0, 24)),
                nn.GroupNorm(5, 25),
                nn.GELU(),
    
                nn.Conv2d(in_channels=25, out_channels=25, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1)),
                nn.GroupNorm(5, 25),
                nn.GELU(),
    
                nn.Conv2d(in_channels=25, out_channels=25, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1)),
                nn.GroupNorm(5, 25),
                nn.GELU(),
            )
        self.spectral_proj = nn.Sequential(
                nn.Linear(101, d_model),
                nn.Dropout(0.1),
            )

    def forward(self, x, mask=None):
        bz, ch_num, patch_num, patch_size = x.shape
        if mask == None:
            mask_x = x
        else:
            mask_x = x.clone()
            mask_x[mask == 1] = self.mask_encoding

        mask_x = mask_x.contiguous().view(bz, 1, ch_num * patch_num, patch_size)
        patch_emb = self.proj_in(mask_x)
        patch_emb = patch_emb.permute(0, 2, 1, 3).contiguous().view(bz, ch_num, patch_num, self.d_model)

        mask_x = mask_x.contiguous().view(bz*ch_num*patch_num, patch_size)
        spectral = torch.fft.rfft(mask_x, dim=-1, norm='forward')
        spectral = torch.abs(spectral).contiguous().view(bz, ch_num, patch_num, 101)
        spectral_emb = self.spectral_proj(spectral)

        patch_emb = patch_emb + spectral_emb
        temporal_emb = self.temporal_encoding(patch_emb.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        spatial_emb = self.spatial_encoding(ch_num)
        patch_emb = patch_emb + temporal_emb + spatial_emb
        return patch_emb